In [ ]:
# Install spacy and its English model
%pip install spacy --quiet
!python -m spacy download en_core_web_sm --quiet

In [6]:
# Install required packages for text preprocessing
%pip install beautifulsoup4 nltk textblob "fuzzywuzzy[speedup]" --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# PodcastIndex Data Collector

Some info about the api we use to collect the data: PodcastIndex 

Collects podcasts two ways (since um... we want a diverse spread of data) then merges them into one deduplicated CSV:

| Source | Endpoint | Received |
|--------|----------|--------------|
| Firehose | `/recent/feeds` | Large broad batch of recently-active podcasts |
| Category search | `/search/byterm` per category from `/categories/list` | Category-tagged podcasts |

Both sources return the same feed object shape, so merging is straightforward.
A podcast found in both gets its categories merged (e.g. `Comedy|Technology`).

**Auth:** SHA-1(`api_key + api_secret + unix_timestamp`) — generated fresh per request.

**Outputs in `data/`:** `podcasts.csv`, `episodes.csv`, `description_embeddings.npy`,
`episode_embeddings.npy`, `embedding_show_ids.txt`, `embedding_episode_ids.txt`, `norm_params.json`

TODO for developers (so yall Estella, Victoria and moi Hannah):

**Setup:** Sign up free at https://api.podcastindex.org/signup: create an `.env` file if you haven't already, and put in
```
PODCASTINDEX_API_KEY="NQPVH4H3DLFV4BZZ5WNX"
PODCASTINDEX_API_SECRET="b$6eMWverPGL^$3VgGkxBhL3naa6z2W6FMgnwAYy"

```

In [1]:
%pip install requests pandas scikit-learn numpy python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests, hashlib, time, os, json
from pathlib import Path
from datetime import datetime
from math import log10
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()
API_KEY    = os.getenv('PODCASTINDEX_API_KEY')
API_SECRET = os.getenv('PODCASTINDEX_API_SECRET')
if not API_KEY or not API_SECRET:
    raise ValueError('Add PODCASTINDEX_API_KEY and PODCASTINDEX_API_SECRET to your .env file.')

BASE_URL   = 'https://api.podcastindex.org/api/1.0'
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

def pi_headers():
    """Fresh SHA-1 auth headers — must be called per request, not cached."""
    ts    = str(int(time.time()))
    token = hashlib.sha1(f'{API_KEY}{API_SECRET}{ts}'.encode()).hexdigest()
    return {'User-Agent': 'PodcastSearchEngine/1.0',
            'X-Auth-Key': API_KEY, 'X-Auth-Date': ts, 'Authorization': token}

def pi_get(endpoint, params=None, retries=3):
    for attempt in range(retries):
        r = requests.get(f'{BASE_URL}{endpoint}', headers=pi_headers(), params=params)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 10))
            print(f'[rate limit] sleeping {wait}s'); time.sleep(wait)
        else:
            print(f'[warn] {r.status_code} {endpoint}: {r.text[:120]}')
            return None
    return None

# Sanity check
stats = pi_get('/stats/current')
if stats:
    s = stats.get('stats', {})
    print(f'Auth OK — index has {s.get("feedCountTotal","?"):,} feeds, {s.get("episodeCountTotal","?"):,} episodes')
else:
    print('Auth FAILED — check credentials.')

Auth OK — index has 4,698,304 feeds, 142,075,934 episodes


In [19]:
# Helper functions

def recency_weight(ts):
    """Unix timestamp of newest episode -> weight for popularity proxy."""
    if not ts: return 0.4
    days = (time.time() - ts) / 86400
    return 1.0 if days <= 90 else (0.7 if days <= 365 else 0.4)

def extract_feed(feed, category=''):
    """
    Flatten a PodcastIndex feed object into a dict.
    `category` is a | -separated string, extended later during merge.

    Key feed fields (confirmed from API docs + python-podcastindex sample responses):
      id, title, url, originalUrl, link, description, author, ownerName,
      image, artwork, lastUpdateTime, lastCrawlTime, lastHttpStatus,
      contentType, itunesId, language, explicit, dead, episodeCount,
      newestItemPublishTime, categories (dict of {id: name})
    """
    ep_count  = feed.get('episodeCount', 0) or 0
    newest_ts = feed.get('newestItemPublishTime', 0) or 0
    # categories field: dict like {'11': 'Comedy', '67': 'Technology'}
    api_cats = feed.get('categories') or {}
    api_cat_str = '|'.join(api_cats.values()) if api_cats else ''
    # Merge with the search-term category if provided and not already present
    merged_cats = merge_categories(api_cat_str, category)

    return {
        'id':                 feed.get('id', ''),
        'podcast_guid':       feed.get('podcastGuid', ''),
        'name':               feed.get('title', ''),
        'author':             feed.get('author', ''),
        'owner_name':         feed.get('ownerName', ''),
        'description':        feed.get('description', ''),
        'categories':         merged_cats,
        'explicit':           bool(feed.get('explicit', 0)),
        'episode_count':      ep_count,
        'language':           feed.get('language', ''),
        'image_url':          feed.get('image', '') or feed.get('artwork', ''),
        'feed_url':           feed.get('url', ''),
        'website_url':        feed.get('link', ''),
        'itunes_id':          feed.get('itunesId', ''),
        'newest_item_date':   datetime.utcfromtimestamp(newest_ts).strftime('%Y-%m-%d') if newest_ts else '',
        'dead':               bool(feed.get('dead', 0)),
        'last_http_status':   feed.get('lastHttpStatus', ''),
        'popularity_score':   round(log10(ep_count + 1) * recency_weight(newest_ts), 4),
        '_source':            '',   # filled in by caller: 'firehose', 'search', or 'firehose|search'
    }

def merge_categories(existing_cats_str, new_cat_str):
    """Merge two |-separated category strings, deduplicating (case-insensitive)."""
    existing = [c for c in existing_cats_str.split('|') if c]
    new      = [c for c in new_cat_str.split('|') if c]
    seen_lower = {c.lower() for c in existing}
    for c in new:
        if c.lower() not in seen_lower:
            existing.append(c)
            seen_lower.add(c.lower())
    return '|'.join(existing)

def merge_into(podcasts_by_id, feed, source_tag, search_category=''):
    """
    Insert feed into podcasts_by_id or merge if already present.
    Returns True if it was a new entry.
    """
    fid = str(feed.get('id', ''))
    if not fid:
        return False
    if fid in podcasts_by_id:
        existing = podcasts_by_id[fid]
        existing['categories'] = merge_categories(existing['categories'], search_category)
        if source_tag not in existing['_source']:
            existing['_source'] = existing['_source'] + '|' + source_tag if existing['_source'] else source_tag
        return False
    else:
        extracted = extract_feed(feed, search_category)
        extracted['_source'] = source_tag
        podcasts_by_id[fid] = extracted
        return True

print('Helpers ready woop woop')

Helpers ready woop woop


---
## Step 1: Firehose — `/recent/feeds`

Grabs large batches of recently-active podcasts by paginating backwards through time
using the `since` parameter (Unix timestamp). Each call returns up to `max` feeds.
We walk back in time until we have enough podcasts or hit our cutoff date.

**Note:** The `categories` field in each feed response is already a dict of `{id: name}` —
so firehose results come pre-categorized with no extra work needed.

In [21]:
# Config — adjust these to control how much to collect
FIREHOSE_TARGET   = 2000   # stop after collecting this many podcasts
FIREHOSE_BATCH    = 100    # max per API call (Note: API supports up to 1000 but 100 is saferrrr)
FIREHOSE_MAX_DAYS = 180    # don't go further back than this many days

podcasts_by_id = {}   # { str(id): dict } — shared across both steps

since_ts  = int(time.time())   #start from current time and increment time backwards
cutoff_ts = since_ts - (FIREHOSE_MAX_DAYS * 86400)
batch_num = 0

print(f'Collecting up to {FIREHOSE_TARGET} podcasts via /recent/feeds...')
print(f'Walking back up to {FIREHOSE_MAX_DAYS} days from now.')

while len(podcasts_by_id) < FIREHOSE_TARGET and since_ts > cutoff_ts:
    data = pi_get('/recent/feeds', params={'max': FIREHOSE_BATCH, 'since': since_ts, 'fulltext': True})
    if not data:
        print('[warn] Empty response, stopping firehose.')
        break

    feeds = data.get('feeds') or []
    if not feeds:
        print('No more feeds returned.')
        break

    new_this_batch = 0
    oldest_ts_in_batch = since_ts

    for feed in feeds:
        if feed is None: continue
        # Track the oldest timestamp in this batch for next pagination step
        last_update = feed.get('lastUpdateTime', 0) or 0
        if last_update and last_update < oldest_ts_in_batch:
            oldest_ts_in_batch = last_update
        if merge_into(podcasts_by_id, feed, 'firehose'):
            new_this_batch += 1

    batch_num += 1
    since_ts = oldest_ts_in_batch - 1   # next batch: go further back

    print(f'  Batch {batch_num}: {new_this_batch} new | {len(podcasts_by_id)} total | '
          f'oldest={datetime.utcfromtimestamp(since_ts).strftime("%Y-%m-%d")}')

    time.sleep(0.5)

print(f'\nFirehose done: {len(podcasts_by_id)} podcasts collected.')

Walking back up to 180 days from now.
No more feeds returned.

Firehose done: 0 podcasts collected.


---
## Step 2: Category search — `/search/byterm` + `/categories/list`

Fetches the official PodcastIndex category list, then runs a `/search/byterm` call
for each category name. Results are merged into `podcasts_by_id`:
- **New podcast:** added with the category name
- **Already seen from firehose:** category is appended to its existing categories string

**Limitation:** `/search/byterm` has no offset/pagination, so `max=500` is the ceiling
per category. The firehose already provides the bulk volume; this step fills in
category coverage.

In [22]:
SEARCH_MAX_PER_CAT = 500   # max results per category search term

# Fetching Categories to search — we will use these category names as search terms in the next step, and also merge them into the podcast metadata for better filtering later.
# This is because PodCastIndex does give us all its categories, but there is no specific api that search podcasts based on these categories id!! Womp womp, so instead we will search with a query that includes the category name, and then merge the category into the podcast metadata for better filtering later.
cat_data = pi_get('/categories/list')
if not cat_data or 'feeds' not in cat_data:
    raise RuntimeError('Could not fetch categories — check auth.')

# Response shape: { 'feeds': [ {'id': 55, 'name': 'Comedy'}, ... ] }
all_categories = {str(c['id']): c['name'] for c in cat_data['feeds']}
print(f'Found {len(all_categories)} categories. Sample:', list(all_categories.values())[:8])

# Set SELECTED_CATS to None to search ALL categories.
SELECTED_CATS = [
    'Comedy', 'True Crime', 'News', 'Technology', 'Sports',
    'Health & Fitness', 'Education', 'Music', 'History', 'Science',
    'Government', 'Business', 'Fiction', 'Society & Culture',
    'Religion & Spirituality', 'Arts', 'Kids & Family',
]

if SELECTED_CATS is None:
    target_cats = all_categories
else:
    sel_lower = {s.lower() for s in SELECTED_CATS}
    target_cats = {cid: name for cid, name in all_categories.items() if name.lower() in sel_lower}
    for s in SELECTED_CATS:
        if s.lower() not in {n.lower() for n in target_cats.values()}:
            print(f'[warn] "{s}" not found in PodcastIndex categories — skipping.')

print(f'\nRunning search for {len(target_cats)} categories...')
before_search = len(podcasts_by_id)

# Searching by category name
for cat_id, cat_name in target_cats.items():
    data = pi_get('/search/byterm', params={'q': cat_name, 'max': SEARCH_MAX_PER_CAT, 'fulltext': True})
    if not data:
        print(f'  [{cat_name}] no response, skipping.')
        continue

    feeds = data.get('feeds') or []
    new_count = sum(1 for f in feeds if f and merge_into(podcasts_by_id, f, 'search', cat_name))
    merged_count = len(feeds) - new_count
    print(f'  [{cat_name}] {len(feeds)} results — {new_count} new, {merged_count} merged into existing')
    time.sleep(0.4)

after_search = len(podcasts_by_id)
print(f'\nSearch done: added {after_search - before_search} new podcasts.')
print(f'Total unique podcasts: {after_search}')

Found 112 categories. Sample: ['Arts', 'Books', 'Design', 'Fashion', 'Beauty', 'Food', 'Performing', 'Visual']
[warn] "Health & Fitness" not found in PodcastIndex categories — skipping.
[warn] "Society & Culture" not found in PodcastIndex categories — skipping.
[warn] "Religion & Spirituality" not found in PodcastIndex categories — skipping.
[warn] "Kids & Family" not found in PodcastIndex categories — skipping.

Running search for 13 categories...
  [Arts] 60 results — 60 new, 0 merged into existing
  [Business] 60 results — 60 new, 0 merged into existing
  [Comedy] 60 results — 60 new, 0 merged into existing
  [Education] 60 results — 59 new, 1 merged into existing
  [Fiction] 60 results — 59 new, 1 merged into existing
  [History] 60 results — 59 new, 1 merged into existing
  [Music] 60 results — 56 new, 4 merged into existing
  [News] 60 results — 57 new, 3 merged into existing
  [Government] 60 results — 60 new, 0 merged into existing
  [Science] 60 results — 54 new, 6 merged into

---
## Step 3: Save merged podcasts to CSV

At this point `podcasts_by_id` contains the full merged set.
The `_source` column shows where each podcast came from: `firehose`, `search`, or `firehose|search`.

In [27]:
df_podcasts = pd.DataFrame(list(podcasts_by_id.values()))

print('Source breakdown:')
print(df_podcasts['_source'].value_counts().to_string())
print()

# Drop dead feeds (no active content)
before = len(df_podcasts)
df_podcasts = df_podcasts[df_podcasts['dead'] == False].reset_index(drop=True)
print(f'Dropped {before - len(df_podcasts)} dead feeds. {len(df_podcasts)} active podcasts remain.')

df_podcasts.to_csv(OUTPUT_DIR / 'podcasts2.csv', index=False)
print(f'Saved -> data/podcasts2.csv ({(OUTPUT_DIR/"podcasts2.csv").stat().st_size/1024:.1f} KB)')

Source breakdown:
_source
search    750

Dropped 0 dead feeds. 750 active podcasts remain.
Saved -> data/podcasts2.csv (714.1 KB)


---
## Step 4: Collect episodes per podcast

Uses `/episodes/byfeedid?id=<id>&max=<n>`.
Saves incrementally every 50 podcasts so progress isn't lost.


### Note: as of 3/18/26, Hannah got 13961 episodes :)

In [28]:
EPISODES_PER_PODCAST = 20
SAVE_EVERY_N         = 50

def extract_episode(ep, feed_id, feed_title, category):
    if ep is None: return None
    dur_sec      = ep.get('duration', 0) or 0
    date_pub     = ep.get('datePublished', 0) or 0
    return {
        'id':             ep.get('id', ''),
        'episode_guid':   ep.get('guid', ''),
        'podcast_id':     feed_id,
        'podcast_name':   feed_title,
        'category':       category,
        'episode_name':   ep.get('title', ''),
        'description':    ep.get('description', ''),
        'duration_sec':   dur_sec,
        'duration_min':   round(dur_sec / 60, 2) if dur_sec else 0,
        'release_date':   datetime.utcfromtimestamp(date_pub).strftime('%Y-%m-%d') if date_pub else '',
        'release_year':   datetime.utcfromtimestamp(date_pub).year if date_pub else None,
        'explicit':       bool(ep.get('explicit', 0)),
        'episode_type':   ep.get('episodeType', ''),
        'season':         ep.get('season', ''),
        'episode_number': ep.get('episode', ''),
        'image_url':      ep.get('image', '') or ep.get('feedImage', ''),
        'audio_url':      ep.get('enclosureUrl', ''),
        'external_url':   ep.get('link', ''),
    }
    
def get_average_episode_duration_for_podcast(feed_id):
    data = pi_get('/episodes/byfeedid', params={'id': feed_id, 'max': EPISODES_PER_PODCAST})
    if not data or 'items' not in data:
        return None
    durations = [ep.get('duration', 0) for ep in data['items'] if ep and ep.get('duration')]
    return round(sum(durations) / len(durations) / 60, 2) if durations else None

# Resume support: skip podcasts we already have episodes for
eps_file = OUTPUT_DIR / 'episodes2.csv'
if eps_file.exists():
    df_existing_eps  = pd.read_csv(eps_file)
    episode_ids_seen = set(df_existing_eps['id'].astype(str))
    done_podcast_ids = set(df_existing_eps['podcast_id'].astype(str))
    all_episodes     = df_existing_eps.to_dict('records')
    print(f'Resuming: {len(all_episodes)} episodes already collected.')
else:
    episode_ids_seen = set()
    done_podcast_ids = set()
    all_episodes     = []

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
to_process  = df_podcasts[~df_podcasts['id'].astype(str).isin(done_podcast_ids)]
print(f'{len(to_process)} podcasts to fetch episodes for.')

for i, row in enumerate(to_process.itertuples(index=False)):
    fid      = str(row.id)
    prim_cat = str(row.categories).split('|')[0]

    data = pi_get('/episodes/byfeedid', params={'id': fid, 'max': EPISODES_PER_PODCAST, 'fulltext': True})
    if data:
        for ep in (data.get('items') or []):
            eid = str((ep or {}).get('id', ''))
            if not eid or eid in episode_ids_seen: continue
            episode_ids_seen.add(eid)
            extracted = extract_episode(ep, fid, row.name, prim_cat)
            if extracted: all_episodes.append(extracted)

    if (i + 1) % SAVE_EVERY_N == 0:
        pd.DataFrame(all_episodes).to_csv(eps_file, index=False)
        print(f'  [{i+1}/{len(to_process)}] {len(all_episodes)} episodes saved...')

    time.sleep(0.3)
    
# add average episode duration to podcast metadata
print('\nCalculating average episode durations for podcasts...')
avg_durations = {}
for i, row in enumerate(df_podcasts.itertuples(index=False)):
    fid = str(row.id)
    avg_dur = get_average_episode_duration_for_podcast(fid)
    avg_durations[fid] = avg_dur
    if (i + 1) % 50 == 0:
        print(f'  Processed {i+1}/{len(df_podcasts)} podcasts...')
df_podcasts['avg_duration_min'] = df_podcasts['id'].astype(str).map(avg_durations)
df_podcasts.to_csv(OUTPUT_DIR / 'podcasts2.csv', index=False)
print(f'Updated podcasts with average episode durations and saved -> data/podcasts2.csv')

df_episodes = pd.DataFrame(all_episodes)
df_episodes.to_csv(eps_file, index=False)
print(f'\nSaved {len(df_episodes)} episodes -> data/episodes2.csv')

750 podcasts to fetch episodes for.
  [50/750] 940 episodes saved...
  [100/750] 1879 episodes saved...
  [150/750] 2841 episodes saved...
  [200/750] 3767 episodes saved...
  [250/750] 4729 episodes saved...
  [300/750] 5587 episodes saved...
  [350/750] 6567 episodes saved...
  [400/750] 7532 episodes saved...
  [450/750] 8383 episodes saved...
  [500/750] 9179 episodes saved...
  [550/750] 10111 episodes saved...
  [600/750] 11051 episodes saved...
  [650/750] 12032 episodes saved...
  [700/750] 12931 episodes saved...
  [750/750] 13930 episodes saved...

Calculating average episode durations for podcasts...
  Processed 50/750 podcasts...
  Processed 100/750 podcasts...
  Processed 150/750 podcasts...
  Processed 200/750 podcasts...
  Processed 250/750 podcasts...
  Processed 300/750 podcasts...
  Processed 350/750 podcasts...
  Processed 400/750 podcasts...
  Processed 450/750 podcasts...
  Processed 500/750 podcasts...
  Processed 550/750 podcasts...
  Processed 600/750 podcasts..

---
## Step 5: Pre-processing 

*Note: (run locally before uploading to container)!!*


This first half of the pre-processing step is to normalize numeric fields (e.g. `total_episodes`, `average_episode_length`) to a 0-1 range using min-max normalization. This ensures that all numeric features are on the same scale, which can help machine learning models converge faster and perform better.
- So for example if you had a podast that was like super popular and had 1 million episodes, and another podcast that was just starting out and had 10 episodes, the raw `total_episodes` feature would be dominated by the popular podcast. But after min-max normalization, both podcasts would have a `total_episodes` value between 0 and 1, which allows the model to learn from both podcasts more effectively.


In [29]:
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes2.csv')

def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-9), float(lo), float(hi)

df_podcasts['episode_count_norm'], ep_lo,  ep_hi  = minmax(df_podcasts['episode_count'].fillna(0))
df_episodes['duration_min_norm'],  dur_lo, dur_hi = minmax(df_episodes['duration_min'].fillna(0))

df_podcasts.to_csv(OUTPUT_DIR / 'podcasts2.csv', index=False)
df_episodes.to_csv(OUTPUT_DIR / 'episodes2.csv', index=False)

with open(OUTPUT_DIR / 'norm_params2.json', 'w') as f:
    json.dump({'episode_count': {'min': ep_lo, 'max': ep_hi},
               'duration_min':  {'min': dur_lo, 'max': dur_hi}}, f, indent=2)

print(f'episode_count range : [{ep_lo:.0f}, {ep_hi:.0f}]')
print(f'duration_min range  : [{dur_lo:.1f}, {dur_hi:.1f}] min')
print('norm_params2.json saved.')

episode_count range : [0, 1000]
duration_min range  : [0.0, 413.4] min
norm_params2.json saved.


The second half is to generate SVD embeddings. Let's keep SVD dimensions to 100

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes2.csv')

# Build text strings: for podcasts, combine name + description + categories; for episodes, combine episode name + description.
def podcast_text(r):
    cats = str(r.get('categories', '')).replace('|', ' ')
    return f"{r['name']} {cats} {r.get('description', '')}"

def ep_text(r):
    return f"{r['episode_name']} {r.get('description', '')}"

show_texts = df_podcasts.apply(podcast_text, axis=1).tolist()
ep_texts   = df_episodes.apply(ep_text,     axis=1).tolist()
show_ids   = df_podcasts['id'].astype(str).tolist()
ep_ids     = df_episodes['id'].astype(str).tolist()

# Podcast SVD embeddings
N_COMPONENTS = 100  # number of SVD dimensions 100-300 for now cuz um thats what i learned

tfidf_shows = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_shows = tfidf_shows.fit_transform(show_texts) # sparse (n_podcasts, 10000)
print(f'Podcast TF-IDF matrix: {tfidf_matrix_shows.shape}')

svd_shows = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
show_embs = svd_shows.fit_transform(tfidf_matrix_shows)  # dense (n_podcasts, 100)
print(f'Explained variance: {svd_shows.explained_variance_ratio_.sum():.2%}')

np.save(OUTPUT_DIR / 'description_embeddings2.npy', show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'embedding_show_ids2.txt').write_text('\n'.join(show_ids))
print(f'Podcast embeddings: shape={show_embs.shape} | {(OUTPUT_DIR/"description_embeddings2.npy").stat().st_size/1024:.0f} KB')

# Episode SVD embeddings:
tfidf_eps = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_eps = tfidf_eps.fit_transform(ep_texts)
print(f'Episode TF-IDF matrix: {tfidf_matrix_eps.shape}')

svd_eps = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
ep_embs = svd_eps.fit_transform(tfidf_matrix_eps)

np.save(OUTPUT_DIR / 'episode_embeddings2.npy', ep_embs.astype(np.float32))
Path(OUTPUT_DIR / 'embedding_episode_ids2.txt').write_text('\n'.join(ep_ids))
print(f'Episode embeddings:  shape={ep_embs.shape} | {(OUTPUT_DIR/"episode_embeddings2.npy").stat().st_size/1024:.0f} KB')

# Save vectorizers + SVD models (needed at query time!) 
import pickle
with open(OUTPUT_DIR / 'svd_shows2.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows}, f)
with open(OUTPUT_DIR / 'svd_episodes2.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_eps, 'svd': svd_eps}, f)
print('Saved svd_shows2.pkl and svd_episodes2.pkl')

Podcast TF-IDF matrix: (750, 10000)
Explained variance: 36.11%
Podcast embeddings: shape=(750, 100) | 293 KB
Episode TF-IDF matrix: (13930, 10000)
Episode embeddings:  shape=(13930, 100) | 5442 KB
Saved svd_shows2.pkl and svd_episodes2.pkl


More SVD preprocessing added:
- stopwords
- punctuation removal
- lowercasing
- normalizing
- using TruncatedSVD to reduce dimensionality of the TF-IDF matrix to 100 components

The cells below are some helper functions to do this: `clean_description` and `generate_svd_embeddings`. The main SVD generation code is in the next cell.

In [31]:
import re
import string
import spacy
from rapidfuzz import process as rapidfuzz_process

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
DEFAULT_STOPWORDS = nlp.Defaults.stop_words | {"podcast", "episode"}

def clean_description(text, vocab=None, stopwords=DEFAULT_STOPWORDS, do_lower=True):
    """
    Preprocess text by:
    - Lowercasing (optional)
    - Removing punctuation
    - Lemmatization
    - Removing stopwords
    If `vocab` is provided, also perform fuzzy matching to correct words to the closest match in the vocab (with a score cutoff of 90).
    """
    if not isinstance(text, str):
        return ''
    if do_lower:
        text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and token.text not in stopwords]
    if vocab:
        words = [
            rapidfuzz_process.extractOne(w, vocab, score_cutoff=90)[0] if rapidfuzz_process.extractOne(w, vocab, score_cutoff=90) else w
            for w in words
        ]
    result = ' '.join(words)
    result = re.sub(r'\s+', ' ', result).strip()
    return result

We will read in `podcasts.csv` and `episodes.csv`, preprocess their description fields once, and create cleaned outputs that will be reused later for SVD embeddings. The cleaned text is saved as `clean_description`. 

In [32]:
INPUT_PATH = OUTPUT_DIR / 'podcasts2.csv'
OUTPUT_PATH =  OUTPUT_DIR / 'podcasts_cleaned2.csv'
CHUNKSIZE = 50 

vocab = set(pd.read_csv(INPUT_PATH, usecols=['description'])['description'].dropna().str.lower().str.split().explode().unique())

first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(lambda x: clean_description(x, vocab))
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} rows...")

print(f"Saved cleaned podcast data to {OUTPUT_PATH}")

INPUT_PATH = OUTPUT_DIR / 'episodes2.csv'
OUTPUT_PATH =  OUTPUT_DIR / 'episodes_cleaned2.csv'
first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(lambda x: clean_description(x, vocab))
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} episode rows...")

print(f"Saved cleaned episode data to {OUTPUT_PATH}")

Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Saved cleaned podcast data to ..\data\podcasts_cleaned2.csv
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 epi

In [5]:
# --- Improved SVD Embedding Pipeline [Testing] ---
import pickle 

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleaned2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned2.csv')

# Build text strings from the cleaned description field and categories.
def podcast_text(r):
    cats = str(r.get('categories', '')).replace('|', ' ')
    return f"{r['name']} {cats} {r.get('clean_description', r.get('description', ''))}"

def ep_text(r):
    return f"{r['episode_name']} {r.get('clean_description', r.get('description', ''))}"

show_texts = df_podcasts.apply(podcast_text, axis=1).tolist()
ep_texts   = df_episodes.apply(ep_text,     axis=1).tolist()
show_ids   = df_podcasts['id'].astype(str).tolist()
ep_ids     = df_episodes['id'].astype(str).tolist()

# Use bigrams and more SVD components
N_COMPONENTS = 100 # WARNING: change this then change other parts too :(
tfidf_shows2 = TfidfVectorizer(max_features=15000, stop_words='english', ngram_range=(1,2), sublinear_tf=True)
tfidf_matrix_shows2 = tfidf_shows2.fit_transform(show_texts)
print(f'Podcast TF-IDF matrix (improved): {tfidf_matrix_shows2.shape}')

svd_shows2 = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
show_embs2 = svd_shows2.fit_transform(tfidf_matrix_shows2)
show_embs2 = normalize(show_embs2, norm='l2')  # L2 normalization
print(f'Explained variance (improved): {svd_shows2.explained_variance_ratio_.sum():.2%}')

# Repeat for episodes
tfidf_eps2 = TfidfVectorizer(max_features=15000, stop_words='english', ngram_range=(1,2), sublinear_tf=True)
tfidf_matrix_eps2 = tfidf_eps2.fit_transform(ep_texts)
print(f'Episode TF-IDF matrix (improved): {tfidf_matrix_eps2.shape}')

svd_eps2 = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
ep_embs2 = svd_eps2.fit_transform(tfidf_matrix_eps2)
ep_embs2 = normalize(ep_embs2, norm='l2')
print(f'Episode explained variance (improved): {svd_eps2.explained_variance_ratio_.sum():.2%}')

# Save improved embeddings and models
np.save(OUTPUT_DIR / 'description_embeddings_improved2.npy', show_embs2.astype(np.float32))
np.save(OUTPUT_DIR / 'episode_embeddings_improved2.npy', ep_embs2.astype(np.float32))
with open(OUTPUT_DIR / 'svd_shows_improved2.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_shows2, 'svd': svd_shows2}, f)
with open(OUTPUT_DIR / 'svd_episodes_improved2.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_eps2, 'svd': svd_eps2}, f)
print('Saved improved SVD embeddings and models.')

Podcast TF-IDF matrix (improved): (750, 15000)
Explained variance (improved): 31.94%
Episode TF-IDF matrix (improved): (13930, 15000)
Episode explained variance (improved): 24.89%
Saved improved SVD embeddings and models.


In [13]:
# Note: Mjght skip
# --- Podcast Embedding: 0.7*episode centroid + 0.3*description embedding ---
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import normalize

# Load episode and podcast description embeddings
show_embs = np.load(OUTPUT_DIR / 'description_embeddings_improved2.npy')  # shape: (n_podcasts, emb_dim)
ep_embs = np.load(OUTPUT_DIR / 'episode_embeddings_improved2.npy')        # shape: (n_episodes, emb_dim)

# Save the old podcast description embeddings before overwriting
# np.save(OUTPUT_DIR / 'old_description_embeddings.npy', show_embs.astype(np.float32))

# Map podcast and episode IDs to their indices
show_id_list = show_ids  # already str
show_id_to_idx = {sid: i for i, sid in enumerate(show_id_list)}
ep_id_list = ep_ids      # already str

df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned2.csv')

# Group episode indices by podcast_id
podcast_to_ep_indices = defaultdict(list)
for i, row in enumerate(df_episodes.itertuples(index=False)):
    podcast_to_ep_indices[str(row.podcast_id)].append(i)

# Compute new podcast embeddings
new_show_embs = np.zeros_like(show_embs)
for sid, show_idx in show_id_to_idx.items():
    ep_indices = podcast_to_ep_indices.get(sid, [])
    if ep_indices:
        centroid = ep_embs[ep_indices].mean(axis=0)
        centroid = normalize(centroid.reshape(1, -1), norm='l2')[0]
        
        new_show_embs[show_idx] = 0.1 * centroid + 0.9 * show_embs[show_idx]
    else:
        new_show_embs[show_idx] = show_embs[show_idx]  # fallback: just description embedding

np.save(OUTPUT_DIR / 'description_embeddings_mixed2.npy', new_show_embs.astype(np.float32))
print('Saved old podcast embeddings to old_description_embeddings.npy')
print('Saved new podcast embeddings (0.1*episode centroid + 0.9*description)')


Saved old podcast embeddings to old_description_embeddings.npy
Saved new podcast embeddings (0.1*episode centroid + 0.9*description)


In [14]:
print(new_show_embs.shape)
print(show_embs.shape)

(750, 100)
(750, 100)


Can skip next code 
We probably need some readable data... so we also save the corresponding podcast and episode IDs in `embedding_show_ids.txt` and `embedding_episode_ids.txt`. This way, we can easily look up which embedding corresponds to which podcast or episode when we need to use them later on.

In [ ]:
# import pandas as pd

# # TF-IDF vocab  -> CSV 
# # The vocabulary maps each word to its column index in the TF-IDF matrix
# vocab_df = pd.DataFrame([
#     {'word': word, 'column_index': idx, 'idf_score': tfidf_shows.idf_[idx]}
#     for word, idx in tfidf_shows.vocabulary_.items()
# ]).sort_values('idf_score')
# vocab_df.to_csv(OUTPUT_DIR / 'tfidf_vocabulary_shows.csv', index=False)
# # Output: word | column_index | idf_score
# # and remember that a Low IDF -> more common word (e.g. "the") and High IDF -> rare/distinctive word

# # SVD components -> CSV 
# # Each row is one SVD dimension, each column is a word to show which words
# # define each "topic" dimension
# components_df = pd.DataFrame(
#     svd_shows.components_, # shape: (n_components, n_vocab)
#     columns=tfidf_shows.get_feature_names_out()
# )
# components_df.index.name = 'svd_dimension'
# components_df.to_csv(OUTPUT_DIR / 'svd_components_shows.csv')
# # Tip:  top words per dimension are prob the "topics" SVD discovered

# #Putting the top words per SVD dimension into a CSV  file
# top_words_per_dim = []
# feature_names = tfidf_shows.get_feature_names_out()
# for dim_idx, component in enumerate(svd_shows.components_):
#     top_indices = component.argsort()[-10:][::-1]  # top 10 words
#     top_words_per_dim.append({
#         'dimension': dim_idx,
#         'top_words': ' | '.join(feature_names[top_indices]),
#         'explained_variance': svd_shows.explained_variance_ratio_[dim_idx],
#     })
# pd.DataFrame(top_words_per_dim).to_csv(OUTPUT_DIR / 'svd_topics_shows.csv', index=False)
# # ^ hopefully use this for um submission or just to learn

In [34]:
# Summary of all the things this notebook has collected woozah
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes2.csv')

print('=' * 55)
print('COLLECTION SUMMARY')
print('=' * 55)
print(f'  Unique podcasts      : {len(df_podcasts)}')
print(f'  Total episodes       : {len(df_episodes)}')
print(f'  Unique categories    : {df_podcasts["categories"].str.split("|").explode().nunique()}')
print(f'  Explicit podcasts    : {df_podcasts["explicit"].sum()}')
print(f'  Source breakdown     :')
print(df_podcasts['_source'].value_counts().to_string())
print(f'  Avg episodes/podcast : {df_podcasts["episode_count"].mean():.1f}')
print(f'  Avg ep duration      : {df_episodes["duration_min"].mean():.1f} min')
print()
print('Output files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = (OUTPUT_DIR / fname).stat().st_size / 1024
    print(f'  {fname:<45} {size_kb:>8.1f} KB')
print()
display(df_podcasts[['name','author','categories','episode_count','popularity_score','_source']].head(8))
display(df_episodes[['podcast_name','episode_name','duration_min','release_date','explicit']].head(5))

COLLECTION SUMMARY
  Unique podcasts      : 750
  Total episodes       : 13930
  Unique categories    : 90
  Explicit podcasts    : 132
  Source breakdown     :
_source
search    750
  Avg episodes/podcast : 291.1
  Avg ep duration      : 41.8 min

Output files:
  categories.json                                    2.3 KB
  description_embeddings2.npy                      293.1 KB
  description_embeddings_improved2.npy             293.1 KB
  embedding_episode_ids2.txt                       172.6 KB
  embedding_show_ids2.txt                            5.8 KB
  embeddings                                         4.0 KB
  episode_embeddings2.npy                         5441.5 KB
  episode_embeddings_improved.npy                10907.2 KB
  episode_embeddings_improved2.npy                5441.5 KB
  episodes.csv                                   26552.1 KB
  episodes2.csv                                  26251.9 KB
  episodes_cleaned2.csv                          42087.1 KB
  norm_params.jso

,name,author,categories,episode_count,popularity_score,_source
0,Talk Art,Russell Tovey and Robert Diament,Arts|Visual,380,1.0324,search
1,The Glenn Beck Program,Mercury Radio Arts,News|Arts,1000,1.2002,search
2,The Art of Charm,The Art of Charm,Education|Business|Health|Fitness|Arts,1000,1.2002,search
3,"Not Me, But You!",Art,Business|Investing|Arts,225,0.9416,search
4,Tales of a Red Clay Rambler: A pottery and cer...,Ben Carter,Arts|Visual|Society|Culture,356,1.0211,search
5,ART IS CHANGE: Strategies & Skills for Activis...,Bill Cleveland,Arts|Society|Culture|News|Politics,173,0.8962,search
6,The Speaking Club: Mastering the Art of Public...,"Sarah Archer: Speaker, Comedian, Author, Playw...",Business|Management|Education|Self Improvement...,333,1.0095,search
7,Art Wank,"Fiona Verity, Julie Nicholson and Gary Seller",Arts|Visual|Business|Entrepreneurship|Educatio...,244,0.9557,search


,podcast_name,episode_name,duration_min,release_date,explicit
0,Talk Art,Lucy Wood on Gwen John,56.17,2026-04-09,False
1,Talk Art,Isaac Julien,59.67,2026-04-02,False
2,Talk Art,Tracey Emin,69.15,2026-03-27,False
3,Talk Art,Collier Schorr,75.12,2026-03-20,False
4,Talk Art,Nicolas Deshayes,66.22,2026-03-13,False
